# Wan 2.2 TI2V-5B: moving shots

Turns each shot's keyframe into a short moving clip (image-to-video),
conditioned on the keyframe image and guided by the shot's `motion_prompt`.

**Before running:** attach a GPU to this notebook (Settings -> Accelerator -> GPU T4 x2),
add your Hugging Face token as a Kaggle Secret named `HF_TOKEN` (Add-ons -> Secrets),
and attach a dataset containing `storyboard.json`, `timing.json` (optional but recommended),
and a `keyframes/` folder with one `{shot_id}.jpg` per shot - zip your local
`storyboard.json` + `outputs/timing.json` + `outputs/keyframes/` together and
create a Kaggle Dataset from that zip, then attach it below.

**Real, confirmed risk:** Wan2.2's own README states TI2V-5B needs >=24GB VRAM
even with every memory-saving flag enabled (its own example GPU is an RTX 4090).
A Kaggle T4 has 16GB. This may out-of-memory. If it does: try a smaller `--size`
(edit the `SIZE` variable below to something like `960*544`, keeping the same
aspect ratio family), or fall back to this project's `--fallback` Ken Burns path,
which needs no GPU at all. Either way, log what you tried in FAILURES.md.

In [ ]:
# --- Configuration: adjust these to match your attached dataset ---
INPUT_DIR = "/kaggle/input/aicontentgeneration"  # <-- change to your dataset's slug
SIZE = "1280*704"  # confirmed 720p setting for ti2v-5B; try "960*544" if this OOMs
OUTPUT_DIR = "/kaggle/working/clips"
FPS = 24  # Wan 2.2's native frame rate

In [ ]:
# --- Install Wan 2.2 ---
# flash_attn frequently fails to build from source on managed notebook images
# (confirmed live: "Getting requirements to build wheel" errored out, which
# aborted the whole `pip install -r requirements.txt` before it got to
# installing ftfy/dashscope - neither of which needs a build step at all,
# and generate.py crashes on a missing ftfy long before any flash_attn-related
# code runs). Installed separately so a flash_attn build failure can't take
# the rest of the requirements down with it.
!git clone https://github.com/Wan-Video/Wan2.2.git
%cd Wan2.2
!pip install -r <(grep -v flash_attn requirements.txt)
!pip install flash_attn --no-build-isolation || echo "flash_attn failed to build - continuing without it"

In [ ]:
# --- Download the Wan2.2-TI2V-5B weights (uses your HF_TOKEN Kaggle Secret) ---
# `huggingface-cli` is deprecated and silently does nothing on current
# huggingface_hub releases (confirmed live: it prints "deprecated and no
# longer works" and the hint text it prints recommends `hf` instead) -
# using that instead.
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!pip install -q -U huggingface_hub
!hf download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B

In [ ]:
# --- Load the storyboard and (if present) narration-driven timing ---
import json

with open(f"{INPUT_DIR}/storyboard.json") as f:
    storyboard = json.load(f)
shots = storyboard["shots"]

timing_by_id = {}
timing_path = f"{INPUT_DIR}/timing.json"
if os.path.exists(timing_path):
    with open(timing_path) as f:
        timing_by_id = {t["id"]: t for t in json.load(f)["shots"]}

print(f"{len(shots)} shots loaded; timing.json {'found' if timing_by_id else 'NOT found - using planned duration_s'}")

In [ ]:
# --- Generate one clip per shot ---
# Resumable: already-rendered shots are skipped, so re-running after a crash
# or a Kaggle session timeout continues where it left off.
import subprocess
import time

os.makedirs(OUTPUT_DIR, exist_ok=True)
gpu_log_path = f"{OUTPUT_DIR}/gpu_log.json"
gpu_log = json.load(open(gpu_log_path)) if os.path.exists(gpu_log_path) else {}


def frames_for_duration(duration_s, fps=FPS):
    # Wan2.2 requires frame_num = 4n + 1
    n = round(duration_s * fps)
    k = round((n - 1) / 4)
    return max(4 * k + 1, 5)


for shot in shots:
    shot_id = shot["id"]
    out_path = f"{OUTPUT_DIR}/{shot_id}.mp4"
    if os.path.exists(out_path):
        print(f"[skip] {shot_id} already rendered")
        continue

    keyframe_path = None
    for ext in ("jpg", "jpeg", "png"):
        candidate = f"{INPUT_DIR}/keyframes/{shot_id}.{ext}"
        if os.path.exists(candidate):
            keyframe_path = candidate
            break
    if keyframe_path is None:
        print(f"[WARN] no keyframe found for {shot_id} (looked for {shot_id}.jpg/.jpeg/.png) - skipping")
        continue

    duration_s = timing_by_id.get(shot_id, {}).get("effective_duration_s", shot["duration_s"])
    frame_num = frames_for_duration(duration_s)
    prompt = shot["motion_prompt"]

    cmd = [
        "python", "generate.py",
        "--task", "ti2v-5B",
        "--size", SIZE,
        "--ckpt_dir", "./Wan2.2-TI2V-5B",
        "--offload_model", "True",
        "--convert_model_dtype",
        "--t5_cpu",
        "--image", keyframe_path,
        "--prompt", prompt,
        "--frame_num", str(frame_num),
        "--save_file", out_path,
    ]
    print(f"Generating {shot_id} ({frame_num} frames, ~{duration_s:.1f}s): {prompt!r}")
    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed_s = time.time() - start

    if result.returncode != 0:
        print(f"[ERROR] {shot_id} failed after {elapsed_s/60:.1f} min:")
        print(result.stderr[-3000:])
        print("If this looks like an out-of-memory error, lower SIZE above (e.g. '960*544') and re-run this cell.")
        continue

    gpu_log[shot_id] = {"gpu_minutes": round(elapsed_s / 60, 2), "size": SIZE, "frame_num": frame_num}
    json.dump(gpu_log, open(gpu_log_path, "w"), indent=2)
    print(f"[OK] {shot_id} done in {elapsed_s/60:.1f} GPU minutes")

print("Done.")

In [ ]:
# --- Zip the clips for download ---
import shutil

shutil.make_archive("/kaggle/working/wan_clips", "zip", OUTPUT_DIR)
print("Saved /kaggle/working/wan_clips.zip - download it from the notebook's Output panel.")